# Preprocessing

In this notebook we normalize and resample the dataset and manifest from the [project setup notebook](./project_setup.ipynb).

In [5]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
WAV16_DIR = PROJECT_ROOT / "data" / "wav16"
MANIFESTS_DIR = PROJECT_ROOT / "manifests"
WAV16_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Manifests dir:", MANIFESTS_DIR)
print("Will write preprocessed WAVs to:", WAV16_DIR)


Project root: /scratch1/ihsiao/Anti-Spoofing-Generalization
Manifests dir: /scratch1/ihsiao/Anti-Spoofing-Generalization/manifests
Will write preprocessed WAVs to: /scratch1/ihsiao/Anti-Spoofing-Generalization/data/wav16


In [6]:
import librosa
import soundfile as sf
import numpy as np
from tqdm import tqdm

TARGET_SR = 16000
PEAK_NORM = 0.99  # peak normalization (max abs value after scaling)

def resample_and_normalize_array(x: np.ndarray, orig_sr: int, target_sr: int = TARGET_SR, peak_norm: float = PEAK_NORM):
    """Return float32 array resampled to target_sr and peak-normalized to peak_norm."""
    if orig_sr != target_sr:
        x = librosa.resample(x, orig_sr=orig_sr, target_sr=target_sr)
    # avoid division by zero
    max_abs = np.max(np.abs(x)) if x.size else 0.0
    if max_abs > 0:
        x = x * (peak_norm / max_abs)
    # ensure dtype float32
    return x.astype(np.float32)

def write_wav(path: Path, array: np.ndarray, sr: int = TARGET_SR):
    """Write array to path as PCM16 WAV using soundfile."""
    path.parent.mkdir(parents=True, exist_ok=True)
    # use subtype PCM_16 for broad compatibility
    sf.write(str(path), array, sr, subtype="PCM_16")


In [7]:
import csv
import pandas as pd

def preprocess_manifest(in_manifest: Path, out_manifest: Path, wav16_dir: Path,
                        target_sr: int = TARGET_SR, peak_norm: float = PEAK_NORM,
                        overwrite: bool = False):
    """
    Read manifest CSV with columns utt,wav,label and:
      - load each audio (librosa supports flac)
      - resample to target_sr, peak-normalize
      - save as wav16_dir/<utt>.wav
      - write out new manifest pointing at preprocessed WAVs (utt,wav,label)
    """
    in_manifest = Path(in_manifest)
    out_manifest = Path(out_manifest)
    wav16_dir = Path(wav16_dir)

    df = pd.read_csv(in_manifest)
    out_rows = []
    failed = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Preprocessing {in_manifest.name}"):
        utt = str(row["utt"])
        orig_wav = Path(row["wav"])
        label = int(row["label"])
        out_wav = wav16_dir / f"{utt}.wav"

        if out_wav.exists() and not overwrite:
            # skip processing if already exists
            out_rows.append((utt, str(out_wav.resolve()), label))
            continue

        try:
            # load file at native sampling rate; librosa returns float32
            y, sr = librosa.load(str(orig_wav), sr=None, mono=True)
            y = resample_and_normalize_array(y, sr, target_sr=target_sr, peak_norm=peak_norm)
            write_wav(out_wav, y, sr=target_sr)
            out_rows.append((utt, str(out_wav.resolve()), label))
        except Exception as e:
            failed.append((utt, str(orig_wav), str(e)))
            # don't append a row for failed files
    # write out new manifest
    out_manifest.parent.mkdir(parents=True, exist_ok=True)
    with open(out_manifest, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["utt", "wav", "label"])
        for r in out_rows:
            writer.writerow(r)

    print(f"Wrote preprocessed manifest: {out_manifest} with {len(out_rows)} entries.")
    if failed:
        print(f"Failed to preprocess {len(failed)} files. Sample failures:")
        for s in failed[:5]:
            print(s)
    return out_manifest


In [8]:
# Manifest files from previous notebook
train_manifest = MANIFESTS_DIR / "LA_train.csv"
dev_manifest   = MANIFESTS_DIR / "LA_dev.csv"
eval_manifest  = MANIFESTS_DIR / "LA_eval.csv"

# Output preprocessed manifests
train_pre = MANIFESTS_DIR / "LA_train_preprocessed.csv"
dev_pre   = MANIFESTS_DIR / "LA_dev_preprocessed.csv"
eval_pre  = MANIFESTS_DIR / "LA_eval_preprocessed.csv"

# Run (set overwrite=True to reprocess)
preprocess_manifest(train_manifest, train_pre, WAV16_DIR, overwrite=False)
preprocess_manifest(dev_manifest, dev_pre, WAV16_DIR, overwrite=False)
preprocess_manifest(eval_manifest, eval_pre, WAV16_DIR, overwrite=False)


Preprocessing LA_train.csv:   0%|          | 0/25380 [00:00<?, ?it/s]/home1/ihsiao/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Preprocessing LA_train.csv: 100%|██████████| 25380/25380 [11:31<00:00, 36.72it/s] 


Wrote preprocessed manifest: /scratch1/ihsiao/Anti-Spoofing-Generalization/manifests/LA_train_preprocessed.csv with 25380 entries.


Preprocessing LA_dev.csv: 100%|██████████| 24844/24844 [10:26<00:00, 39.65it/s]


Wrote preprocessed manifest: /scratch1/ihsiao/Anti-Spoofing-Generalization/manifests/LA_dev_preprocessed.csv with 24844 entries.


Preprocessing LA_eval.csv: 100%|██████████| 71237/71237 [1:14:02<00:00, 16.04it/s] 


Wrote preprocessed manifest: /scratch1/ihsiao/Anti-Spoofing-Generalization/manifests/LA_eval_preprocessed.csv with 71237 entries.


PosixPath('/scratch1/ihsiao/Anti-Spoofing-Generalization/manifests/LA_eval_preprocessed.csv')

In [9]:
import pandas as pd
from IPython.display import Audio, display

def manifest_stats(manifest_path):
    df = pd.read_csv(manifest_path)
    total = len(df)
    bon = int((df.label == 0).sum())
    spoof = int((df.label == 1).sum())
    durations = []
    # sample up to 20 files to compute durations quickly
    for wav in df.wav.sample(min(20, total), random_state=1).values:
        try:
            y, sr = librosa.load(wav, sr=None)
            durations.append(len(y)/sr)
        except Exception:
            pass
    avg_dur = sum(durations)/len(durations) if durations else None
    return {"total": total, "bonafide": bon, "spoof": spoof, "avg_sampled_duration_s": avg_dur}

for m in [train_pre, dev_pre, eval_pre]:
    stats = manifest_stats(m)
    print(m.name, stats)

# Play one random preprocessed audio from the training manifest
df_train = pd.read_csv(train_pre)
sample_row = df_train.sample(1, random_state=42).iloc[0]
print("Playing", sample_row['utt'], sample_row['wav'])
display(Audio(sample_row['wav']))


LA_train_preprocessed.csv {'total': 25380, 'bonafide': 2580, 'spoof': 22800, 'avg_sampled_duration_s': 3.5800937500000005}
LA_dev_preprocessed.csv {'total': 24844, 'bonafide': 2548, 'spoof': 22296, 'avg_sampled_duration_s': 3.35255}
LA_eval_preprocessed.csv {'total': 71237, 'bonafide': 7355, 'spoof': 63882, 'avg_sampled_duration_s': 3.273978125}
Playing LA_T_1136756 /scratch1/ihsiao/Anti-Spoofing-Generalization/data/wav16/LA_T_1136756.wav


### Note

Frames here refers to sample count.

In [10]:
import soundfile as sf

# check first few WAVs
df = pd.read_csv(MANIFESTS_DIR / "LA_train_preprocessed.csv")
for wav in df.wav.head(5).values:
    info = sf.info(wav)
    print(Path(wav).name, "samplerate:", info.samplerate, "subtype:", info.subtype, "frames:", info.frames)


LA_T_1138215.wav samplerate: 16000 subtype: PCM_16 frames: 55329
LA_T_1271820.wav samplerate: 16000 subtype: PCM_16 frames: 70323
LA_T_1272637.wav samplerate: 16000 subtype: PCM_16 frames: 46392
LA_T_1276960.wav samplerate: 16000 subtype: PCM_16 frames: 45001
LA_T_1341447.wav samplerate: 16000 subtype: PCM_16 frames: 56531
